In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F 
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

import numpy as np 
import matplotlib.pyplot as plt 
import matplotlib.pylab as plt2
import os

from models import *

# from resnet50 import ResNet50
# from resnet34 import ResNet34

from torch.optim.lr_scheduler import ReduceLROnPlateau




In [2]:
#Check GPU, connect to it if it is available 
device = ''
if torch.cuda.is_available():
	device = 'cuda'
	print("CUDA is available. GPU will be used for training.")
else:
	device = 'cpu'


BEST_ACCURACY = 0

# Preparing Data
print("==> Prepairing data ...")
#Transformation on train data
transform_train = transforms.Compose([
	transforms.RandomCrop(32, padding=4),
	transforms.RandomHorizontalFlip(),
	transforms.ToTensor(),
	transforms.Normalize((0.4914, 0.4822, 0.4465),(0.2023, 0.1994, 0.2010)),
	])

#transformation on validation data
transform_validation = transforms.Compose([
	transforms.ToTensor(),
	transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
	])

#Download Train and Validation data and apply transformation
train_data = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
validation_data = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_validation)

#Put data into trainloader, specify batch_size
train_loader = torch.utils.data.DataLoader(train_data, batch_size=128, shuffle=True, num_workers=2)
validation_loader = torch.utils.data.DataLoader(validation_data, batch_size=128, shuffle=True, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

#Function to show CIFAR images
def show_data(image):
	plt.imshow(np.transpose(image[0], (1, 2, 0)), interpolation='bicubic')
	plt.show()

#show_data(train_data[0])




==> Prepairing data ...
Files already downloaded and verified
Files already downloaded and verified


In [34]:
#Need to import model a model
model = ResNet34_class()

# checkpoint = torch.load('./Results/resnet34/trained/backbone_transfer.pth')
checkpoint = torch.load('./Results/resnet34/trained/backbone_34_trained.pth', map_location=torch.device('cpu'))
model.load_state_dict(checkpoint)

for name, param in model.named_parameters():
    param.requires_grad = False

model = ResNet34_x(model)

#model = CNN_batch()
#Pass model to GPU
model = model.to(device)
model.train()

for name, param in model.named_parameters():
#     if param.requires_grad:
    print(f'Layer: {name}, Shape: {param.data.shape}')
    print(param.data)

Layer: backbone.conv1.weight, Shape: torch.Size([64, 3, 3, 3])
tensor([[[[ 6.0265e-02, -2.5049e-01, -4.1791e-02],
          [-2.5547e-01, -1.0566e+00, -5.0522e-01],
          [-6.2602e-02, -4.4846e-01, -9.2616e-02]],

         [[-2.2530e-02,  3.1884e-01,  1.0311e-02],
          [ 3.3075e-01,  1.1291e+00,  4.8684e-01],
          [-6.6306e-03,  4.2044e-01,  1.3416e-01]],

         [[-4.4745e-02, -7.2696e-02,  3.9060e-02],
          [-8.0943e-02, -1.9795e-02, -4.6264e-02],
          [ 7.0330e-02, -2.1382e-02, -1.0477e-02]]],


        [[[-2.9665e-05, -2.9382e-05, -3.1984e-05],
          [-3.2967e-05, -2.6357e-05, -3.1429e-05],
          [-3.3632e-05, -2.6590e-05, -3.5100e-05]],

         [[-4.2823e-05, -5.8260e-05, -5.2632e-05],
          [-4.4985e-05, -5.6062e-05, -4.3042e-05],
          [-5.1598e-05, -4.5908e-05, -5.1924e-05]],

         [[-3.4740e-05, -2.4660e-05, -3.3691e-05],
          [-3.6056e-05, -2.4534e-05, -3.1877e-05],
          [-3.1966e-05, -3.5296e-05, -3.2282e-05]]],


   

tensor([[[[-3.0728e-03,  2.9738e-03,  2.6998e-03],
          [ 6.3677e-03,  5.2609e-03,  4.4353e-03],
          [ 1.0234e-02,  7.0637e-04, -1.1101e-03]],

         [[-4.2811e-03, -1.2784e-03, -4.6531e-03],
          [ 8.3172e-05,  4.4305e-03,  3.1859e-04],
          [ 6.1454e-03,  7.3364e-03,  3.7865e-03]],

         [[ 2.9092e-04,  1.0154e-03,  4.8476e-04],
          [ 4.5535e-03,  3.6200e-03,  1.4596e-03],
          [ 5.3577e-04, -2.7642e-03, -1.3199e-03]],

         ...,

         [[ 7.0500e-04, -1.3703e-03,  1.7431e-03],
          [ 1.0047e-02,  8.9410e-03,  1.1902e-02],
          [ 1.3661e-02,  1.4881e-02,  1.6484e-02]],

         [[-1.1622e-02, -8.4363e-03, -6.0053e-03],
          [-1.5215e-02, -1.3441e-02, -7.3808e-03],
          [-3.6951e-03, -2.2168e-03, -7.0597e-04]],

         [[ 3.8880e-04,  3.5894e-03,  1.7074e-04],
          [-1.0872e-02, -1.2633e-02, -1.0698e-02],
          [-1.5128e-02, -1.6320e-02, -1.1277e-02]]],


        [[[-5.7454e-03,  1.2469e-03,  9.1360e-03],
  

In [29]:

optimizer = optim.SGD(model.parameters(), lr = 0.01, momentum=0.9, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=5, verbose=True)



length_train = len(train_data)
length_validation = len(validation_data)
#print(length_train)
#print(len(train_loader))
num_classes = 10

In [30]:
#Training
def train(epochs):
	global BEST_ACCURACY
	dict = {'Train Loss':[], 'Train Acc':[], 'Validation Loss':[], 'Validation Acc':[]}
	for epoch in range(epochs):
		print("\nEpoch:", epoch+1, "/", epochs)
		cost = 0
		correct = 0
		total = 0
		woha = 0

		for i, (x,y) in enumerate(train_loader):
            
			woha += 1
			model.train()
			x, y = x.to(device), y.to(device)
			optimizer.zero_grad()
			yhat = model(x)
			yhat = yhat.reshape(-1, 10)
			loss = criterion(yhat, y)
			loss.backward()
			optimizer.step()
			cost += loss.item()

			_, yhat2 = torch.max(yhat.data, 1)
			correct += (yhat2 == y).sum().item()
			total += y.size(0)
			print("\nAcc:", correct/total, correct, "/", total)

		my_loss = cost/len(train_loader)
		my_accuracy = 100*correct/length_train

		dict['Train Loss'].append(my_loss)
		dict['Train Acc'].append(my_accuracy)

		print('Tain Loss:', my_loss)
		print('Train Accuracy:', my_accuracy,'%')


		cost = 0
		correct = 0

		with torch.no_grad():
			for x, y in validation_loader:
				x, y = x.to(device), y.to(device)
				model.eval()
				yhat = model(x)
				yhat = yhat.reshape(-1, 10)
				loss = criterion(yhat, y)
				cost += loss.item()
				
				_, yhat2 = torch.max(yhat.data, 1)
				correct += (yhat2 == y).sum().item()

		my_loss = cost/len(validation_loader)
		my_accuracy = 100*correct/length_validation

		dict['Validation Loss'].append(my_loss)
		dict['Validation Acc'].append(my_accuracy)

		print('Validation Loss:', my_loss)
		print('Validation Accuracy:', my_accuracy,'%')
        
		# Update the optimizer parameters
		optimizer.step()

		# Update the scheduler based on the validation accuracy
		scheduler.step(my_accuracy)

		# Zero the gradients
		optimizer.zero_grad()

		#Save the model if you get best accuracy on validation data
		if my_accuracy > BEST_ACCURACY:
			BEST_ACCURACY = my_accuracy
			print('Saving the model ...')
			model.eval()
			if not os.path.isdir('checkpoint'):
			    os.mkdir('checkpoint')
			torch.save(model.state_dict(), './checkpoint/trained/resnet34.pth')

	print("TRAINING IS FINISHED !!!")
	return dict



In [31]:
#Start training
results = train(10)


# plt.figure(1)
# plt.plot(results['Train Loss'], 'b', label = 'training loss')
# plt.plot(results['Validation Loss'], 'r', label = 'validation loss')
# plt.title("LOSS")
# plt.xlabel("Epochs")
# plt.ylabel("Loss")
# plt.legend(['training set', 'validation set'], loc='center right')
# plt.savefig('Loss_ResNet50.png', dpi=300, bbox_inches='tight')

# plt.figure(2)
# plt.plot(results['Train Acc'], 'b', label = 'training accuracy')
# plt.plot(results['Validation Acc'], 'r', label = 'validation accuracy')
# plt.title("ACCURACY")
# plt.xlabel("Epochs")
# plt.ylabel("Accuracy")
# plt.legend(['training set', 'validation set'], loc='center right')
# plt.savefig('Accuracy_ResNet50.png', dpi=300, bbox_inches='tight')
# plt.show()
# plt.close()

"""
axs[0].plot(results['Train Loss'], 'b', label = 'training loss')
axs[0].plot(results['Validation Loss'], 'r', label = 'validation loss')
axs[0].set_title("LOSS")
axs[0].set(xlabel="Epochs", ylabel="Loss")

axs[1].plot(results['Train Acc'], 'b', label = 'training accuracy')
axs[1].plot(results['Validation Acc'], 'r', label = 'validation accuracy')
axs[1].set_title("ACCURACY")
axs[1].set(xlabel="Epochs", ylabel="Accuracy")

fig.tight_layout()
plt.legend()
plt.show()
"""



Epoch: 1 / 2

Acc: 0.1171875 15 / 128

Acc: 0.09765625 25 / 256

Acc: 0.11979166666666667 46 / 384

Acc: 0.109375 56 / 512

Acc: 0.1109375 71 / 640

Acc: 0.109375 84 / 768

Acc: 0.10825892857142858 97 / 896

Acc: 0.1103515625 113 / 1024

Acc: 0.11631944444444445 134 / 1152

Acc: 0.11875 152 / 1280

Acc: 0.12144886363636363 171 / 1408

Acc: 0.125 192 / 1536

Acc: 0.125 208 / 1664

Acc: 0.12834821428571427 230 / 1792

Acc: 0.13229166666666667 254 / 1920

Acc: 0.13623046875 279 / 2048

Acc: 0.13924632352941177 303 / 2176

Acc: 0.14322916666666666 330 / 2304

Acc: 0.14473684210526316 352 / 2432

Acc: 0.14765625 378 / 2560

Acc: 0.15141369047619047 407 / 2688

Acc: 0.15696022727272727 442 / 2816

Acc: 0.1610054347826087 474 / 2944

Acc: 0.16243489583333334 499 / 3072

Acc: 0.1665625 533 / 3200

Acc: 0.1700721153846154 566 / 3328

Acc: 0.17100694444444445 591 / 3456

Acc: 0.17243303571428573 618 / 3584

Acc: 0.17295258620689655 642 / 3712

Acc: 0.17604166666666668 676 / 3840

Acc: 0.1784274


Acc: 0.44632393973214285 12797 / 28672

Acc: 0.44645833333333335 12858 / 28800

Acc: 0.4471446349557522 12935 / 28928

Acc: 0.4479281387665198 13015 / 29056

Acc: 0.44836211622807015 13085 / 29184

Acc: 0.4489628820960699 13160 / 29312

Acc: 0.44938858695652173 13230 / 29440

Acc: 0.4498782467532468 13302 / 29568

Acc: 0.4503973599137931 13375 / 29696

Acc: 0.45064377682403434 13440 / 29824

Acc: 0.45092147435897434 13506 / 29952

Acc: 0.45159574468085106 13584 / 30080

Acc: 0.4518008474576271 13648 / 30208

Acc: 0.452103111814346 13715 / 30336

Acc: 0.4528295693277311 13795 / 30464

Acc: 0.45309231171548114 13861 / 30592

Acc: 0.4539713541666667 13946 / 30720

Acc: 0.45484310165975106 14031 / 30848

Acc: 0.4553202479338843 14104 / 30976

Acc: 0.45608281893004116 14186 / 31104

Acc: 0.45635886270491804 14253 / 31232

Acc: 0.4567283163265306 14323 / 31360

Acc: 0.4572217987804878 14397 / 31488

Acc: 0.45767965587044535 14470 / 31616

Acc: 0.4582913306451613 14548 / 31744

Acc: 0.458521


Acc: 0.6112847222222222 3521 / 5760

Acc: 0.6126019021739131 3607 / 5888

Acc: 0.6128656914893617 3687 / 6016

Acc: 0.6124674479166666 3763 / 6144

Acc: 0.6128826530612245 3844 / 6272

Acc: 0.61375 3928 / 6400

Acc: 0.6150428921568627 4015 / 6528

Acc: 0.6150841346153846 4094 / 6656

Acc: 0.6155660377358491 4176 / 6784

Acc: 0.6147280092592593 4249 / 6912

Acc: 0.6140625 4323 / 7040

Acc: 0.6114676339285714 4383 / 7168

Acc: 0.6110197368421053 4458 / 7296

Acc: 0.611260775862069 4538 / 7424

Acc: 0.6101694915254238 4608 / 7552

Acc: 0.609375 4680 / 7680

Acc: 0.6095030737704918 4759 / 7808

Acc: 0.6082409274193549 4827 / 7936

Acc: 0.6094990079365079 4915 / 8064

Acc: 0.6087646484375 4987 / 8192

Acc: 0.6081730769230769 5060 / 8320

Acc: 0.6080729166666666 5137 / 8448

Acc: 0.6079757462686567 5214 / 8576

Acc: 0.6089154411764706 5300 / 8704

Acc: 0.6090353260869565 5379 / 8832

Acc: 0.6087053571428571 5454 / 8960

Acc: 0.609375 5538 / 9088

Acc: 0.6083984375 5607 / 9216

Acc: 0.607876


Acc: 0.6196233365019012 20859 / 33664

Acc: 0.6195549242424242 20936 / 33792

Acc: 0.6196049528301887 21017 / 33920

Acc: 0.6195664943609023 21095 / 34048

Acc: 0.6195283239700374 21173 / 34176

Acc: 0.6197819496268657 21261 / 34304

Acc: 0.6200917750929368 21351 / 34432

Acc: 0.6202256944444444 21435 / 34560

Acc: 0.6201856549815498 21513 / 34688

Acc: 0.6200884650735294 21589 / 34816

Acc: 0.6200778388278388 21668 / 34944

Acc: 0.6202953923357665 21755 / 35072

Acc: 0.6203125 21835 / 35200

Acc: 0.6200181159420289 21904 / 35328

Acc: 0.6201207129963899 21987 / 35456

Acc: 0.6199696492805755 22061 / 35584

Acc: 0.6201556899641577 22147 / 35712

Acc: 0.6203125 22232 / 35840

Acc: 0.6203013790035588 22311 / 35968

Acc: 0.6205673758865248 22400 / 36096

Acc: 0.620389796819788 22473 / 36224

Acc: 0.6204885563380281 22556 / 36352

Acc: 0.620422149122807 22633 / 36480

Acc: 0.6206020541958042 22719 / 36608

Acc: 0.6204540505226481 22793 / 36736

Acc: 0.6207682291666666 22884 / 36864

Acc: 

'\naxs[0].plot(results[\'Train Loss\'], \'b\', label = \'training loss\')\naxs[0].plot(results[\'Validation Loss\'], \'r\', label = \'validation loss\')\naxs[0].set_title("LOSS")\naxs[0].set(xlabel="Epochs", ylabel="Loss")\n\naxs[1].plot(results[\'Train Acc\'], \'b\', label = \'training accuracy\')\naxs[1].plot(results[\'Validation Acc\'], \'r\', label = \'validation accuracy\')\naxs[1].set_title("ACCURACY")\naxs[1].set(xlabel="Epochs", ylabel="Accuracy")\n\nfig.tight_layout()\nplt.legend()\nplt.show()\n'

In [32]:
torch.save(model.backbone.state_dict(), './Results/resnet34/trained/backbone_transfer_1.pth')
torch.save(model.linear.state_dict(), './Results/resnet34/trained/linear_transfer_1.pth')

In [33]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f'Layer: {name}, Shape: {param.data.shape}')
        print(param.data)

Layer: linear.weight, Shape: torch.Size([10, 512, 1, 1])
tensor([[[[-0.0108]],

         [[ 0.0517]],

         [[-0.1302]],

         ...,

         [[-0.1106]],

         [[-0.0272]],

         [[ 0.1992]]],


        [[[-0.0016]],

         [[ 0.0195]],

         [[ 0.1383]],

         ...,

         [[-0.0592]],

         [[ 0.0183]],

         [[-0.1685]]],


        [[[ 0.0294]],

         [[-0.0069]],

         [[-0.0807]],

         ...,

         [[ 0.0394]],

         [[ 0.1745]],

         [[-0.1519]]],


        ...,


        [[[-0.0500]],

         [[ 0.2509]],

         [[-0.0130]],

         ...,

         [[-0.0697]],

         [[-0.0766]],

         [[-0.1207]]],


        [[[-0.0609]],

         [[-0.0839]],

         [[-0.0972]],

         ...,

         [[-0.0431]],

         [[-0.0366]],

         [[-0.0027]]],


        [[[ 0.0701]],

         [[-0.0663]],

         [[-0.0208]],

         ...,

         [[-0.0228]],

         [[-0.1562]],

         [[-0.0396]]]])

In [ ]:
# Save the layers except the linear layer
layer_dict = {}
for name, layer in model.named_children():
    if name != 'linear':
        layer_dict[name] = layer.state_dict()

# Save the linear layer separately
linear_dict = model.linear.state_dict()

torch.save(layer_dict, './Results/resnet34/backbone_34_new.pth')
torch.save(linear, './Results/resnet34/linear_34_new.pth')